# D14. Do patients share Stage 1 factor distributions?

**Purpose.** The diagnostic suite (D3) asks whether a panel's *predictive* lift is shared or patient-specific. It does not answer the aggregation question: after factors are learned on everyone, do patients occupy the same score distributions, or do they sit on disjoint supports that should be treated as separate subtypes?

This notebook runs that test on stored Stage 1 scores. It does not refit PCA, factor analysis, or FactoSig.

## What is already done

Stage 1 fit each basis on all 60,118 eligible cells from 13 patients. `scores.npy` is the projection of every cell onto that basis. Comparing per-patient score distributions is the check. A new DR fit is not required.

## What this test is not

- Not a cell-level Kolmogorov–Smirnov p-value. Cells inside a patient are dependent, and with thousands of cells that test rejects trivial shifts.
- Not D3. A large paired patient-specific AUPRC gap can coexist with overlapping factor supports.
- Not a joint overlap in 40 dimensions. In that dimension, overlap collapses for every cohort. The estimand is per-factor marginal overlap, variance-weighted across factors.
- Not an inductive deployment check. The basis saw every patient.


## Estimand and decision rule

Within a stratum (normal cells, or malignant cells), eligible patients are those with at least 200 normal cells or at least 20 malignant cells. The malignant stratum is the seven supported patients. P05, P08, P10, P11, P12, and P13 do not have enough malignant cells to enter it.

For each factor, histogram overlap (the overlapping coefficient) uses one 32-bin grid on the included cells' 1st–99th percentile, with tails placed in the end bins. The factor summary is the median overlap across patient pairs. The panel summary is the variance-weighted mean of those medians.

The null shuffles patient labels inside the stratum, keeps each patient's cell count fixed, and keeps the bins fixed. The reported gap is null overlap minus observed overlap. With 25 permutations the null is extremely tight, so a lower-tail p-value hits the resolution floor whenever the gap is larger than about 0.01. The decision uses the overlap magnitude, not that p-value.

| Call | Rule |
| --- | --- |
| `aggregate_shared` | Normal weighted overlap ≥ 0.75 and every patient's normal-cell overlap with the others ≥ 0.60, and malignant weighted overlap ≥ 0.45 with gap < 0.15 |
| `separate_subtype_candidate` | Some patient's normal-cell overlap with the others < 0.40, or malignant weighted overlap < 0.35 and gap ≥ 0.15 |
| `shared_with_location_shift` | Neither. Exchangeability can fail while the supports still overlap |

A second pass flags a single factor, rather than the average, when malignant-cell η² ≥ 0.75, variance share ≥ 0.15, and median pairwise overlap < 0.50. That is the check for a patient-private axis hiding inside an overlapping average.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

REPO = Path("/home/minhang/mds_project/sc_classification")
MODULE = REPO / "scripts" / "mrd_stage0_2" / "stage2_supervised"
if str(MODULE) not in sys.path:
    sys.path.insert(0, str(MODULE))

import patient_factor_overlap as pfo

RECOMPUTE = False
TABLE_DIR, FIGURE_DIR = pfo.output_dirs(REPO)
calls_path = TABLE_DIR / "patient_factor_overlap_calls.csv"

if RECOMPUTE or not calls_path.exists():
    tables = pfo.run_screen(REPO, n_perm=pfo.N_PERM_DEFAULT)
    pfo.write_screen(tables, REPO)
    factors, celltypes = pfo.focus_factor_table(REPO, n_perm=pfo.N_PERM_DEFAULT)
    factors.to_csv(TABLE_DIR / "focus_factor_overlap.csv", index=False)
    celltypes.to_csv(TABLE_DIR / "focus_celltype_overlap.csv", index=False)
    pfo.low_overlap_factors(REPO).to_csv(TABLE_DIR / "low_overlap_high_variance_factors.csv", index=False)
    pfo.write_rule_card(REPO)

calls = pd.read_csv(calls_path)
flags = pd.read_csv(TABLE_DIR / "low_overlap_high_variance_factors.csv")
print(TABLE_DIR)
print(calls.aggregation_call.value_counts().to_string())
print("representations", len(calls))


In [ ]:
biological = calls.loc[calls.atlas_row_block.eq("biological")].copy()
summary = biological[
    [
        "short_panel_label",
        "stage1_method",
        "effective_k",
        "normal_weighted_overlap",
        "normal_overlap_gap",
        "normal_min_patient_vs_rest",
        "normal_min_patient_id",
        "cancer_weighted_overlap",
        "cancer_overlap_gap",
        "cancer_min_patient_id",
        "aggregation_call",
        "provisional_three_way_taxonomy",
    ]
].sort_values("cancer_weighted_overlap")
summary.to_csv(TABLE_DIR / "biological_overlap_summary.csv", index=False)
display(summary.round(3))

In [ ]:
pfo.save_overlap_figures(REPO)
findings_path = pfo.write_findings(REPO)
print(findings_path)
print((TABLE_DIR / "patient_factor_overlap_findings.md").read_text())


## Result

Forty of 41 biological panels are `aggregate_shared`. Normal-cell weighted overlap is 0.78–0.89 (median 0.84). Malignant-cell weighted overlap is 0.54–0.66 (median 0.61). The malignant gap versus the permutation null has median 0.06. Patients are not exchangeable, and they are not separate subtypes on these bases.

Allograft rejection is the one biological `shared_with_location_shift`. Its averages are 0.85 (normal) and 0.62 (malignant). P02's normal cells overlap the other patients at 0.58, just under the 0.60 minimum-patient gate. D14C shows that bar. The supports still overlap.

D3's patient-specific tags do not line up with this test. Panels tagged `patient_only_candidate` or `shared_plus_patient_specific_candidate` still have overlapping factor scores. A predictive gap is not a disjoint factor support.

The exception is outside the 41 programs. On the D1-selected broad bags, one malignant factor isolates P03:

- HVG 10,000 FactoSig K=40, factor 21: 37% of malignant score variance, η² 0.91, P03 overlap with the other malignant cells 0.007.
- HVG 3,000 FA K=40, factor 27: P03 overlap 0.013.
- All Stage 0 gene sets, FactoSig K=40, factor 30: P03 overlap 0.009.

No screened Hallmark, Reactome, or group-panel factor meets that η² and variance-share pair. Within normal cell types, the overlap gap on the focus bases stays at or below 0.10 (D14E), so the normal-cell patient shift is not a hidden cell-type subtype.

![D14A panel overlap](../../../../experiments/20260525_060508_stage0_mrd_old34_broad_screen_82db5093/analysis/figures/stage2_diagnostic/patient_factor_overlap_v1/D14A_panel_factor_overlap.png)

![D14B permutation gap](../../../../experiments/20260525_060508_stage0_mrd_old34_broad_screen_82db5093/analysis/figures/stage2_diagnostic/patient_factor_overlap_v1/D14B_overlap_gap_versus_permutation.png)

![D14C focus patients](../../../../experiments/20260525_060508_stage0_mrd_old34_broad_screen_82db5093/analysis/figures/stage2_diagnostic/patient_factor_overlap_v1/D14C_focus_patient_vs_rest_overlap.png)

![D14D score distributions](../../../../experiments/20260525_060508_stage0_mrd_old34_broad_screen_82db5093/analysis/figures/stage2_diagnostic/patient_factor_overlap_v1/D14D_factor_score_distributions.png)

![D14E cell type](../../../../experiments/20260525_060508_stage0_mrd_old34_broad_screen_82db5093/analysis/figures/stage2_diagnostic/patient_factor_overlap_v1/D14E_celltype_overlap_gap.png)


## Take-home

Aggregate patients when reading the biological Stage 1 bases. The factor distributions overlap. Treat a factor as patient-private only when its own support splits, which in this screen happens for P03 malignant cells on three broad-bag factors and not on the 41 program bases.

Do not convert a significant permutation test into a subtype split. Every normal-cell comparison rejected exchangeability, and the observed overlap stayed above 0.75.

## Claim boundary

This is a descriptive overlap test on a transductive atlas. It does not say that a shared classifier transfers, that personalization has no benefit, or that P03's broad-bag factor is a biological subtype rather than a donor, composition, or technical axis. Malignant overlap is estimated from 22–758 cells per patient, and the small-sample null itself sits near 0.65–0.70, so malignant overlap in the high 0.50s is close to what shared distributions produce at this sample size.


## D15. Held-out AUPRC versus number of training cells

**Estimand.** On the same frozen Stage 1 scores as D14, refit only the atlas-selected Stage 2 probe on a label-stratified random subset of the training cells. The held-out patient is scored in full, so AUPRC stays comparable as the training set grows. Three draws are averaged at each size below the full set.

**What this is not.** Not a refit of PCA, FA, or FactoSig. Those factors already saw every cell, including the held-out patient and the cells left out of a draw. Not a curve for sequencing a new patient. Not a retune of `C`: the penalty is the one selected on the full cohort. Every draw of 500 cells still contains all 12 training patients, so the x-axis is cell depth at a fixed patient mix. At 500 training cells the draw has about 13 malignant cells; at the full set, about 1,500.

**Panels.** Interferon-α PCA K=40 and FA K=40 (both selected L1, C=1e-4), apoptosis FactoSig-promax K=40 (L2, C=1e-4), and allograft rejection FactoSig-promax K=40 (elastic-net). The full-training refit matches the D1 atlas patient AUPRCs (maximum absolute difference below 0.0002).


In [ ]:
import cell_count_learning_curve as cclc

RECOMPUTE_CURVE = False
curve_dir, _figure_dir = cclc.output_dirs(REPO)
curve_path = curve_dir / "cell_count_learning_curve.csv"
if RECOMPUTE_CURVE or not curve_path.exists():
    curve = cclc.run_curve(REPO)
    cclc.write_curve(curve, REPO)
    cclc.save_curve_figures(curve, REPO)
else:
    curve = pd.read_csv(curve_path)

agreement = cclc.atlas_agreement(curve, REPO)
print(f"full-training refit vs atlas AUPRC: max |diff| = {agreement['abs_auprc_diff'].max():.4g}")
summary = cclc.summarize_curve(curve)
view = summary.assign(
    median_auprc=summary["median_auprc"].round(3),
    median_lift_pp=(100 * summary["median_lift"]).round(2),
    p20_lift_pp=(100 * summary["p20_lift"]).round(2),
    p80_lift_pp=(100 * summary["p80_lift"]).round(2),
    n_train_cells=summary["n_train_cells"].round(0).astype(int),
    n_train_malignant=summary["n_train_malignant"].round(0).astype(int),
)[
    [
        "curve_label",
        "is_full_training",
        "n_train_cells",
        "n_train_malignant",
        "median_auprc",
        "median_lift_pp",
        "p20_lift_pp",
        "p80_lift_pp",
    ]
]
display(view)


## Reading

Apoptosis already has most of its cohort lift at about 1,000 training cells. Median lift is 7.7 pp at 500 cells, 9.3 pp at 1,000, and 11.5 pp on the full training set, which is the atlas number. At 1,000 cells, P02 / P03 / P04 are within a few points of their full-data lifts (32 vs 40, 48 vs 47, 21 vs 18 pp).

Allograft rejection stays at prevalence through 2,000 cells, then rises: median lift 1.7 pp at 5,000, 3.1 pp at 10,000, 8.8 pp at 25,000, and 12.6 pp on the full set.

Interferon-α does not. With the atlas L1 setting (C=1e-4), both PCA K=40 and FA K=40 sit on each patient's prevalence at every subsample through 25,000 cells. An all-zero probe has AUPRC equal to prevalence, so lift is 0. The atlas pattern appears only on the full training set: PCA median lift 11.1 pp, carried by P02, P03, P04, and P07; FA median lift stays 0 because only P03 and P04 move. The same small C collapsing to an all-zero solution is the donor-downsampling result (H-S2-04). It is not evidence that 25,000 cells contain no interferon signal. A `C` chosen for that sample size could differ, and this curve does not retune `C`.

![D15A AUPRC vs training cells](../../../../experiments/20260525_060508_stage0_mrd_old34_broad_screen_82db5093/analysis/figures/stage2_diagnostic/cell_count_learning_curve_v1/D15A_auprc_vs_training_cells.png)

![D15B median lift vs training cells](../../../../experiments/20260525_060508_stage0_mrd_old34_broad_screen_82db5093/analysis/figures/stage2_diagnostic/cell_count_learning_curve_v1/D15B_median_lift_vs_training_cells.png)
